# Reproduce the competition submission

This notebook downloads the public code and exact prepared seasonal snapshots, checks your official competition inputs, fits the final model, and verifies the submitted CSV hash. It does not issue a new monthly forecast.

Before running: attach the **official WorCAP competition dataset** with **Add Input**, enable Internet, and choose a CPU session with enough RAM (the historical run used Kaggle). No GPU or CDS secret is needed. The external snapshots are about 17 MB. The official dataset is about 2 GB compressed and is obtained from Kaggle separately.

Only the final fit runs by default. The seven historical validation tables remain archived evidence; they are not recomputed here. Run all cells in order in a fresh session.


In [ ]:
from pathlib import Path
import json, subprocess, sys

REPO = Path('/kaggle/working/worcap-reproduction')
REVISION = 'data-snapshots-v1'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REVISION,
                    'https://github.com/Houxxie/worcap-2026-rainfall-forecasting.git', str(REPO)], check=True)
else:
    current = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
    expected = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', REVISION + '^{commit}'], text=True).strip()
    if current != expected or subprocess.check_output(['git', '-C', str(REPO), 'diff', '--name-only'], text=True).strip():
        raise RuntimeError('Existing checkout differs. Use a fresh session or a new REPO directory.')
print('Code revision:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'competition/requirements.txt')], check=True)


## Prepare and check the inputs

Only the prepared SEAS5 and CFSv2 fields are downloaded from GitHub. The official dataset stays in Kaggle Input. This step verifies all historical file hashes and the core package versions before fitting anything.


In [ ]:
subprocess.run([sys.executable, str(REPO / 'scripts/download_data.py'), '--output', str(REPO / 'data')], check=True)
candidates = sorted({p.parent.resolve() for p in Path('/kaggle/input').rglob('treino_tp.nc')})
if len(candidates) != 1:
    raise RuntimeError('Attach exactly one official WorCAP dataset using Add Input; found ' + str(len(candidates)))
config = REPO / 'configs/kaggle.local.json'
config.write_text(json.dumps({'profile': 'competition', 'inputs': {
    'official': str(candidates[0]), 'seas5': str(REPO / 'data/seas5'), 'cfsv2': str(REPO / 'data/cfsv2')
}}, indent=2), encoding='utf-8')
subprocess.run([sys.executable, str(REPO / 'scripts/reproduce_competition.py'), '--config', str(config), '--check-only'], check=True)


## Fit the final model and reproduce the CSV

The final model is trained on 1993–2022. Feature order, weights, input bytes and intermediate CSV rounding follow the recorded submission. A different final hash stops the run; no tolerance is used to claim byte-for-byte identity.


In [ ]:
OUTPUT = Path('/kaggle/working/competition-reproduction')
subprocess.run([sys.executable, str(REPO / 'scripts/reproduce_competition.py'), '--config', str(config), '--output', str(OUTPUT)], check=True)


## Result

Keep the generated CSV and its identity audit together. This is a reproduction check, not a new evaluation on unknown rainfall observations.


In [ ]:
from IPython.display import FileLink, display
identities = sorted(OUTPUT.rglob('identidade_csv.json'))
if not identities:
    raise RuntimeError('No completed CSV identity audit found.')
latest = identities[-1]
audit = json.loads(latest.read_text(encoding='utf-8'))
if audit.get('identico_byte_a_byte') is not True:
    raise RuntimeError('CSV differs from the recorded submission.')
print(json.dumps(audit, indent=2))
display(FileLink(str(latest.parent / 'submission_hibrida.csv')))
